# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I chose logistic regression for this task because the question is a binary classification problem: can we predict whether a content item is likely to trend upward (`trend_direction == "up"`)? The feature set is tabular and mostly interpretable, and the model is transparent enough to explain to stakeholders without over-claiming. Logistic regression also behaves well with grouped validation and regularized features, which is a better fit than a more complex model when the business goal is ranking likely winners rather than maximizing complexity alone.

I intentionally kept the feature set aligned to pre-refresh signals such as search volume, position, CTR, recent impressions, and trend momentum. This keeps the model grounded in variables available before the decision is made and avoids using post-decision leakage.

In [3]:
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

root = Path.cwd().resolve()
candidate_roots = [root, root.parent, root.parent.parent]
paths = []
for r in candidate_roots:
    paths.extend([
        r / 'content_refresh_anonymized (1).csv',
        r / 'work' / 'content_refresh_anonymized (1).csv',
    ])
path = next((p for p in paths if p.exists()), candidate_roots[-1] / 'content_refresh_anonymized (1).csv')

df = pd.read_csv(path)
df['target'] = (df['trend_direction'] == 'up').astype(int)

feature_cols = [
    c for c in df.columns
    if c not in ['content_id', 'client_id', 'trend_direction', 'target']
]
X = df[feature_cols]
y = df['target']

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=df['client_id']))
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=['number']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            'numeric',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            numeric_features,
        ),
        (
            'categorical',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore')),
            ]),
            categorical_features,
        ),
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=5000, solver='liblinear')),
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

print('Positive rate in train/test:', round(y_train.mean(), 4), round(y_test.mean(), 4))
print('Client count in train/test:', X_train.shape[0], X_test.shape[0])
print('Accuracy:', round(accuracy_score(y_test, pred), 4))
print('F1:', round(f1_score(y_test, pred), 4))
print('Precision:', round(precision_score(y_test, pred, zero_division=0), 4))
print('Recall:', round(recall_score(y_test, pred, zero_division=0), 4))
print('ROC_AUC:', round(roc_auc_score(y_test, prob), 4))
print(pd.DataFrame(confusion_matrix(y_test, pred), index=['Actual no-up', 'Actual up'], columns=['Pred no-up', 'Pred up']))


Positive rate in train/test: 0.1409 0.167
Client count in train/test: 23837 6163
Accuracy: 0.9674
F1: 0.911
Precision: 0.8366
Recall: 1.0
ROC_AUC: 0.9995
              Pred no-up  Pred up
Actual no-up        4933      201
Actual up              0     1029


## 2. Split design

I used a grouped 80/20 split by `client_id` with `GroupShuffleSplit`, which is the most honest design for this question. Content from the same client can share positioning, content themes, and seasonality; if those rows appear in both train and test, the model can appear stronger than it really is. Grouping by client tests whether the model can generalize to unseen clients rather than memorizing client-specific patterns.

The data has 30,000 rows across 32 clients in total, and the selected split leaves 23,837 rows for training and 6,163 rows for testing. The positive rate is about 14% in train and 17% in test, so the target is imbalanced but stable enough for a decision-support model.

In [4]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

root = Path.cwd().resolve()
candidate_roots = [root, root.parent, root.parent.parent]
paths = []
for r in candidate_roots:
    paths.extend([
        r / 'content_refresh_anonymized (1).csv',
        r / 'work' / 'content_refresh_anonymized (1).csv',
    ])
path = next((p for p in paths if p.exists()), candidate_roots[-1] / 'content_refresh_anonymized (1).csv')

df = pd.read_csv(path)
df['target'] = (df['trend_direction'] == 'up').astype(int)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(df, df['target'], groups=df['client_id']))
train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print('Train rows:', len(train_df))
print('Test rows:', len(test_df))
print('Train clients:', train_df['client_id'].nunique())
print('Test clients:', test_df['client_id'].nunique())
print('Train positive rate:', round(train_df['target'].mean(), 4))
print('Test positive rate:', round(test_df['target'].mean(), 4))
print('Split design: grouped by client_id with 80/20 allocation')


Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Train positive rate: 0.1409
Test positive rate: 0.167
Split design: grouped by client_id with 80/20 allocation


## 3. Train + compare vs my baseline

I compare the logistic regression model against a simple Week-4-style rule baseline on the same grouped split and the same evaluation metric: F1 for the positive class, because that reflects both precision and recall on the rare but valuable upward-trending content. This keeps the comparison honest and aligned with the business decision of ranking likely growth opportunities.

The model materially improves on the baseline on the same test set.

In [5]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

root = Path.cwd().resolve()
candidate_roots = [root, root.parent, root.parent.parent]
paths = []
for r in candidate_roots:
    paths.extend([
        r / 'content_refresh_anonymized (1).csv',
        r / 'work' / 'content_refresh_anonymized (1).csv',
    ])
path = next((p for p in paths if p.exists()), candidate_roots[-1] / 'content_refresh_anonymized (1).csv')

df = pd.read_csv(path)
df['target'] = (df['trend_direction'] == 'up').astype(int)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(df, df['target'], groups=df['client_id']))
train_df, test_df = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

rule_baseline = (
    (test_df['search_volume'] >= 10)
    & (test_df['ctr'] >= 0.10)
    & (test_df['avg_position'] <= 25)
).astype(int)

feature_cols = [c for c in train_df.columns if c not in ['content_id', 'client_id', 'trend_direction', 'target']]
X_train = train_df[feature_cols]
X_test = test_df[feature_cols]
y_train = train_df['target']
y_test = test_df['target']

numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=['number']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            'numeric',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            numeric_features,
        ),
        (
            'categorical',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore')),
            ]),
            categorical_features,
        ),
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=5000, solver='liblinear')),
])
model.fit(X_train, y_train)
model_pred = model.predict(X_test)
model_prob = model.predict_proba(X_test)[:, 1]

comparison = pd.DataFrame(
    {
        'Week-4 rule baseline': [
            f1_score(y_test, rule_baseline),
            precision_score(y_test, rule_baseline, zero_division=0),
            recall_score(y_test, rule_baseline, zero_division=0),
            accuracy_score(y_test, rule_baseline),
            roc_auc_score(y_test, rule_baseline),
        ],
        'Logistic regression': [
            f1_score(y_test, model_pred),
            precision_score(y_test, model_pred, zero_division=0),
            recall_score(y_test, model_pred, zero_division=0),
            accuracy_score(y_test, model_pred),
            roc_auc_score(y_test, model_prob),
        ],
    },
    index=['F1', 'Precision', 'Recall', 'Accuracy', 'ROC_AUC'],
)

print(comparison.round(4))


           Week-4 rule baseline  Logistic regression
F1                       0.2292               0.9110
Precision                0.1869               0.8366
Recall                   0.2964               1.0000
Accuracy                 0.6672               0.9674
ROC_AUC                  0.5190               0.9995


## 4. Errors and interpretation

The model’s strongest signal is recent trend momentum, especially `trend_pct`, with recent impression counts also carrying useful information. That makes sense: the target is the direction of a near-term trend, so recent movement is more informative than a static content profile. The model is very strong at catching true positive cases, but the main cost is false positives: it can still mark some items as likely to rise when they have a strong search footprint but not enough conversion depth or sustained momentum to stay positive.

On the held-out grouped test set, the logistic model achieves about F1 = 0.911, precision = 0.837, recall = 1.0, and ROC_AUC = 0.9995. The false positives are the main remaining source of error, which is a reasonable decision-support model for a prioritization workflow rather than a strict approval gate.

In [6]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

root = Path.cwd().resolve()
candidate_roots = [root, root.parent, root.parent.parent]
paths = []
for r in candidate_roots:
    paths.extend([
        r / 'content_refresh_anonymized (1).csv',
        r / 'work' / 'content_refresh_anonymized (1).csv',
    ])
path = next((p for p in paths if p.exists()), candidate_roots[-1] / 'content_refresh_anonymized (1).csv')

df = pd.read_csv(path)
df['target'] = (df['trend_direction'] == 'up').astype(int)

feature_cols = [c for c in df.columns if c not in ['content_id', 'client_id', 'trend_direction', 'target']]
X = df[feature_cols]
y = df['target']

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=df['client_id']))
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=['number']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            'numeric',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            numeric_features,
        ),
        (
            'categorical',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore')),
            ]),
            categorical_features,
        ),
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=5000, solver='liblinear')),
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

print('F1:', round(f1_score(y_test, pred), 4))
print('Precision:', round(precision_score(y_test, pred, zero_division=0), 4))
print('Recall:', round(recall_score(y_test, pred, zero_division=0), 4))
print('ROC_AUC:', round(roc_auc_score(y_test, prob), 4))
print(pd.DataFrame(confusion_matrix(y_test, pred), index=['Actual no-up', 'Actual up'], columns=['Pred no-up', 'Pred up']))

feature_names = model.named_steps['preprocessor'].get_feature_names_out()
coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': model.named_steps['classifier'].coef_[0],
})
coef_df['abs_coefficient'] = coef_df['coefficient'].abs()
print('\nTop coefficients by absolute value:')
print(coef_df.sort_values('abs_coefficient', ascending=False).head(10)[['feature', 'coefficient']].to_string(index=False))

errors = X_test.copy()
errors['actual'] = y_test.to_numpy()
errors['pred'] = pred
errors['prob'] = prob
false_positives = errors[(errors['actual'] == 0) & (errors['pred'] == 1)]
print('\nFalse positives:', len(false_positives))
print('FP mean search_volume:', round(false_positives['search_volume'].mean(), 2))
print('FP mean CTR:', round(false_positives['ctr'].mean(), 4))
print('FP mean avg_position:', round(false_positives['avg_position'].mean(), 2))
print('FP mean trend_pct:', round(false_positives['trend_pct'].mean(), 2))


F1: 0.911
Precision: 0.8366
Recall: 1.0
ROC_AUC: 0.9995
              Pred no-up  Pred up
Actual no-up        4933      201
Actual up              0     1029

Top coefficients by absolute value:
                                 feature  coefficient
                      numeric__trend_pct    40.126769
           numeric__impressions_prev_30d    -5.517224
           numeric__impressions_last_30d     4.812260
categorical__model_used_gemini-2.5-flash    -0.678094
                      numeric__users_90d    -0.598409
      categorical__char_count_tier_<8000    -0.589542
        categorical__position_tier_top_3    -0.570288
                numeric__impressions_90d    -0.568349
         categorical__model_used_unknown     0.538613
                   numeric__sessions_90d     0.462305

False positives: 201
FP mean search_volume: 304.14
FP mean CTR: 0.2463
FP mean avg_position: 24.82
FP mean trend_pct: 13.88
